In [ ]:
import os
import sys

# SCRIPT_PATH = "/home/yifanhou/git/universal_manipulation_interface/scripts"
# sys.path.append(os.path.join(SCRIPT_PATH, '../'))
# sys.path.append(os.path.join(SCRIPT_PATH, '../../'))
sys.path.append("../../")

import numpy as np
import plotly.graph_objs as go
import plotly.io as pio
import zarr
from plotly.offline import init_notebook_mode
from plotly.subplots import make_subplots
from PyriteUtility.computer_vision.imagecodecs_numcodecs import register_codecs

register_codecs()

pio.templates.default = "plotly_dark"
pio.renderers.default = "vscode"
init_notebook_mode(connected=True)


if "PYRITE_DATASET_FOLDERS" not in os.environ:
    raise ValueError("Please set the environment variable PYRITE_DATASET_FOLDERS")
dataset_folder_path = os.environ.get("PYRITE_DATASET_FOLDERS")

dataset_path = dataset_folder_path + "/drag_highres_perturb/"

store = zarr.DirectoryStore(path=dataset_path)
buffer = zarr.open(store=store, mode="a")

# buffer.tree()

In [ ]:
# for key in buffer.keys():
#     print(key)

for ep, epdata in buffer["data"].items():
    for key in epdata.keys():
        print(key)
    break

In [ ]:
from PyriteUtility.math.numerical_differentiation import finite_difference

count = 0
for ep, ep_data in buffer["data"].items():
    # count += 1
    # if count < 72:
    #     continue
    print(ep)

    # episode_1722317514
    # camera0_rgb
    # low_dim_time_stamps
    # stiffness
    # ts_pose_command
    # ts_pose_fb
    # ts_pose_virtual_target
    # visual_time_stamps
    # wrench
    # wrench_filtered

    ts_pose_fb = ep_data["ts_pose_fb_0"][:]
    ts_pose_command = ep_data["ts_pose_command_0"][:]
    # ts_pose_virtual_target = ep_data["ts_pose_virtual_target"][:]
    wrench = ep_data["wrench_0"][:]
    wrench_filtered = ep_data["wrench_filtered_0"][:]
    robot_times = ep_data["robot_time_stamps_0"][:]
    wrench_times = ep_data["wrench_time_stamps_0"][:]

    # compute velocity and acc, then filter with moving average
    ts_vel_fb = np.diff(ts_pose_fb, axis=0) / np.diff(robot_times)[:, None]
    ts_acc_fb = np.diff(ts_vel_fb, axis=0) / np.diff(robot_times[1:])[:, None]
    window = 15
    print("Computing moving average")
    ts_vel_ave = np.zeros_like(ts_vel_fb)
    ts_acc_ave = np.zeros_like(ts_acc_fb)
    # fmt: off
    ts_vel_ave[:, 0] = np.convolve(ts_vel_fb[:, 0], np.ones(window) / window, mode="same")
    ts_vel_ave[:, 1] = np.convolve(ts_vel_fb[:, 1], np.ones(window) / window, mode="same")
    ts_vel_ave[:, 2] = np.convolve(ts_vel_fb[:, 2], np.ones(window) / window, mode="same")
    ts_acc_ave[:, 0] = np.convolve(ts_acc_fb[:, 0], np.ones(window) / window, mode="same")
    ts_acc_ave[:, 1] = np.convolve(ts_acc_fb[:, 1], np.ones(window) / window, mode="same")
    ts_acc_ave[:, 2] = np.convolve(ts_acc_fb[:, 2], np.ones(window) / window, mode="same")
    # fmt: on
    ts_vel_fb = ts_vel_ave
    ts_acc_fb = ts_acc_ave

    # # Compute velocity and acc with numerial differentiation
    # N = ts_pose_fb.shape[0]
    # ts_vel_fb = finite_difference(ts_pose_fb[:, :3], 5+np.arange(N-10), 0.002, 1)
    # ts_acc_fb = finite_difference(ts_pose_fb[:, :3], 5+np.arange(N-10), 0.002, 2)

    # # compute wrench in the world frame
    # N = ts_pose_fb.shape[0]
    # wrench_W = np.zeros((N, 6))
    # wrench_filtered_W = np.zeros((N, 6))
    # for i in range(N):
    #     pose7_WT = ts_pose_fb[i]
    #     wrenchi_T = wrench[i]
    #     wrenchi_filtered_T = wrench_filtered[i]
    #     SE3_WT = SE3.Rt(q2r(pose7_WT[3:7]), pose7_WT[0:3], check=False)
    #     SE3_TW = SE3_WT.inv()
    #     wrench_W[i, :] = SE3_TW.Ad().T @ wrenchi_T
    #     wrench_filtered_W[i, :] = SE3_TW.Ad().T @ wrenchi_filtered_T

    # delta_pose = ts_pose_virtual_target - ts_pose_fb

    fig = make_subplots(
        rows=4,
        cols=3,
        shared_xaxes=True,
        subplot_titles=(
            "X",
            "Y",
            "Z",
            "dX",
            "dY",
            "dZ",
            "ddX",
            "ddY",
            "ddZ",
            "fx",
            "fy",
            "fz",
        ),
    )

    fig.add_trace(
        go.Scatter(x=robot_times, y=ts_pose_fb[:, 0], name="ts_pose_fb0"), row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=robot_times, y=ts_pose_fb[:, 1], name="ts_pose_fb1"), row=1, col=2
    )
    fig.add_trace(
        go.Scatter(x=robot_times, y=ts_pose_fb[:, 2], name="ts_pose_fb2"), row=1, col=3
    )

    fig.add_trace(
        go.Scatter(x=robot_times[1:], y=ts_vel_fb[:, 0], name="ts_vel_fb0"),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=robot_times[1:], y=ts_vel_fb[:, 1], name="ts_vel_fb1"),
        row=2,
        col=2,
    )
    fig.add_trace(
        go.Scatter(x=robot_times[1:], y=ts_vel_fb[:, 2], name="ts_vel_fb2"),
        row=2,
        col=3,
    )

    fig.add_trace(
        go.Scatter(x=robot_times[2:], y=ts_acc_fb[:, 0], name="ts_acc_fb0"),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=robot_times[2:], y=ts_acc_fb[:, 1], name="ts_acc_fb1"),
        row=3,
        col=2,
    )
    fig.add_trace(
        go.Scatter(x=robot_times[2:], y=ts_acc_fb[:, 2], name="ts_acc_fb2"),
        row=3,
        col=3,
    )

    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 0], name="wrench0"), row=4, col=1
    )
    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 1], name="wrench1"), row=4, col=2
    )
    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 2], name="wrench2"), row=4, col=3
    )

    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench_filtered[:,0], name='wrench_filtered0'),row=3, col=1)
    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench_filtered[:,1], name='wrench_filtered1'),row=3, col=2)
    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench_filtered[:,2], name='wrench_filtered2'),row=3, col=3)

    fig.update_layout(height=900, width=1900, title_text=ep)
    fig.show()
    # fig.write_html('output.html')

    input("Press Enter to continue...")

In [ ]:
# plot wrench changes

count = 0
for ep, ep_data in buffer["data"].items():
    print(ep)

    wrench = ep_data["wrench_0"][:]
    wrench_times = ep_data["wrench_time_stamps_0"][:]

    # dn = 700 # 100ms
    dn = 70  # 10ms
    # dn = 35 # 5ms
    wrench_diff = wrench[dn:] - wrench[:-dn]
    wrench_diff_times = wrench_times[dn:]
    wrench_diff_norm = np.linalg.norm(wrench_diff[:, :3], axis=1)

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, subplot_titles=("FX", "FY", "FZ", "dF")
    )

    fig.add_trace(
        go.Scatter(x=wrench_diff_times, y=wrench_diff_norm, name="wrench diff norm"),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 0], name="wrench0"), row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 1], name="wrench1"), row=3, col=1
    )
    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 2], name="wrench2"), row=4, col=1
    )

    fig.update_layout(height=900, width=1900, title_text=ep)
    fig.show()
    # fig.write_html('output.html')

    # input("Press Enter to continue...")
    break

In [ ]:
# Test the augmentation on simulated data

import copy
import os
import sys

sys.path.append("/home/yifanhou/git/PyriteML")
sys.path.append("../../")

from PyriteML.diffusion_policy.common.sampler_with_augmentations import (
    augment_k,
    fit_polynomial_k,
)
from tqdm import tqdm

# parameter
# M*xdd = F + K*(x_bar - x) + D*xd
M = np.array([5, 5, 5])
K = np.eye(3) * 0  # 1000
D = np.eye(3) * 3


N = 100000
Nchange = 40000
dts = 0.00001


# initial condition and control
x0 = np.zeros(3)
f = np.ones([N, 3])
f[Nchange:] = -1
x = np.zeros([N, 3])

X_bar = np.zeros([N, 3])
# # generate brownian motion in X_bar
# for t in range(N-1):
#     X_bar[t+1] = X_bar[t] + np.random.normal(0, 0.0002, 3)
# # constant X_bar
# X_bar = 0.02*np.ones([N, 3])
# # Sinosoidal X_bar
# X_bar = 0.02*np.sin(np.arange(N) * 0.0002)[:, None] * np.array([1, 1, 0])


# simulate
x[0] = x0
xd = np.zeros(3)
for t in tqdm(range(N - 1)):
    Mxdd = f[t] + K @ (X_bar[t] - x[t]) + D @ xd
    xdd = Mxdd / M
    xd += xdd * dts
    x[t + 1] = x[t] + xd * dts

# sample data
dt = 0.002
x_sampled = x[:: int(dt / dts)]
X_bar_sampled = X_bar[:: int(dt / dts)]  # debugging purpose only
f_sampled = f[:: int(dt / dts)]

N_sampled = x_sampled.shape[0]

Nstart = int(0.35 * N_sampled)  # if this is changed, make sure Nchange is updated too
Nend = int(0.45 * N_sampled)

x_piece = x_sampled[Nstart:Nend]
f_piece = f_sampled[Nstart:Nend]
X_bar_piece = X_bar_sampled[Nstart:Nend]

# add noise
x_piece += 0.0001 * (np.random.random(x_piece.shape) - 0.5)
f_piece += 0.5 * (np.random.random(f_piece.shape) - 0.5)

r_horizon = Nend - Nstart

## Step one: compute the primary direction of variation of this piece
vector = np.array([1, 0, 0])  # if changed, gt parameters need to be changed
vector = vector / np.linalg.norm(vector)
## Step two: compute the projection of pos wrench on this vector
twist_proj = np.sum(x_piece * vector, axis=1)
wrench_proj = np.sum(f_piece * vector, axis=1)
# Step three: solve for a1, a2, b0, b1, b2
k = 20
para = fit_polynomial_k(wrench_proj, twist_proj, k)
# dN = 1
# a1, a2, b1, b2 = fit_polynomial_a1a2b1b2(wrench_proj, twist_proj, dN = dN, method="least_square")
# a1, a2, b1, b2 = fit_polynomial_a1a2b1b2(wrench_proj, twist_proj, method="qp")

# step four: augment data in the projected space
# Faug = [0]
Faug = [0, 10]
# Faug = [0, 1, 2, 5]
# wrench_proj_aug_all, twist_proj_aug_all = augment_a1a2b1b2(
#     a1, a2, b1, b2,
#     wrench_proj,
#     twist_proj,
#     Faug,
#     dN=dN
# )
wrench_proj_aug_all, twist_proj_aug_all = augment_k(
    para,
    k,
    wrench_proj,
    twist_proj,
    Faug,
)

# compute ground truth
start_id = Nstart * int(dt / dts)
x_gt = copy.copy(x)
xd = np.zeros(3)
for t in tqdm(range(N - 1)):
    if t >= start_id:
        f_gt = f[t] + Faug[1] * vector
    else:
        f_gt = f[t]

    Mxdd = f_gt + K @ (X_bar[t] - x_gt[t]) + D @ xd
    xdd = Mxdd / M
    xd += xdd * dts
    x_gt[t + 1] = x_gt[t] + xd * dts
# gt parameters
l0 = 1 + (K[0, 0] + D[0, 0] / dt) * dt * dt / M[0]
l1 = dt * dt / M[0]
l2 = 2 + dt * D[0, 0] / M[0]
l3 = dt * dt / M[0] * K[0, 0] * X_bar[start_id, 0]
a1_gt = -l2 / l0
a2_gt = 1 / l0
b1_gt = -l1 / l0
b2_gt = -l3 / l0

# sample data
x_gt_sampled = x_gt[:: int(dt / dts)]
x_gt_piece = x_gt_sampled[Nstart:Nend]
# projection
twist_proj_gt = np.sum(x_gt_piece * vector, axis=1)
# additional plotting
X_bar_proj_gt = np.sum(X_bar_piece * vector, axis=1)

# illustrate the result
# print(f"a1:{a1} ({a1_gt}), a2:{a2} ({a2_gt}), b1:{b1} ({b1_gt}), b2:{b2} ({b2_gt}).")
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("twist projection", "force projection"),
)
fig.add_trace(
    go.Scatter(
        x=np.arange(r_horizon),
        y=X_bar_proj_gt,
        name="X_bar",
        mode="lines",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.arange(r_horizon),
        y=twist_proj,
        name="original",
        mode="markers",
    ),
    row=1,
    col=1,
)
for f in range(len(Faug)):
    fig.add_trace(
        go.Scatter(
            x=np.arange(r_horizon),
            y=twist_proj_aug_all[f],
            name=f"aug_prediction{f}",
            mode="markers",
        ),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(
        x=np.arange(r_horizon),
        y=twist_proj_gt,
        name="aug_gt",
        mode="markers",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.arange(r_horizon),
        y=wrench_proj,
        name="wrench_proj",
        mode="markers",
    ),
    row=2,
    col=1,
)
for f in range(len(Faug)):
    fig.add_trace(
        go.Scatter(
            x=np.arange(r_horizon),
            y=wrench_proj_aug_all[f],
            name=f"wrench_proj_aug_{f}",
            mode="markers",
        ),
        row=2,
        col=1,
    )
fig.update_layout(height=600, width=800, title_text="augmented data")
fig.show()
print("Done for one")


# # plot
# time = np.arange(N) * dt
# fig = make_subplots(
#     rows=2, cols=1,
#     shared_xaxes=True, subplot_titles=('F', 'X'))

# fig.add_trace(go.Scatter(x=time, y=x[:, 0], name='x0'),row=1, col=1)
# fig.add_trace(go.Scatter(x=time, y=f[:, 0], name='f0'),row=2, col=1)

# fig.update_layout(height=900, width=1900, title_text=ep)
# fig.show()

In [ ]:
# test augmentation on real data

import copy
import os
import sys

sys.path.append("/home/yifanhou/git/PyriteML")
sys.path.append("../../")


from PyriteML.diffusion_policy.common.sampler_with_augmentations import (
    augment_k,
    compute_variation_score,
    fit_polynomial_k,
)
from PyriteUtility.spatial_math.spatial_utilities import (
    SE3_inv,
    SE3_to_spt,
    pose7_to_SE3,
)
from tqdm import tqdm

init_notebook_mode(connected=True)


for ep, ep_data in buffer["data"].items():
    print(ep)

    ts_pose_fb = ep_data["ts_pose_fb_0"][:]
    ts_pose_command = ep_data["ts_pose_command_0"][:]
    # ts_pose_virtual_target = ep_data["ts_pose_virtual_target"][:]
    wrench = ep_data["wrench_0"][:]
    wrench_filtered = ep_data["wrench_filtered_0"][:]
    robot_times = ep_data["robot_time_stamps_0"][:]
    wrench_times = ep_data["wrench_time_stamps_0"][:]

    # normalize the quaternion portion of ts_pose_fb
    for i in range(ts_pose_fb.shape[0]):
        ts_pose_fb[i, 3:7] = ts_pose_fb[i, 3:7] / np.linalg.norm(ts_pose_fb[i, 3:7])
    SE3_fb = pose7_to_SE3(ts_pose_fb)

    # # compute velocity and acc, then filter with moving average
    # ts_vel_fb = np.diff(ts_pose_fb, axis=0) / np.diff(robot_times)[:, None]
    # ts_acc_fb = np.diff(ts_vel_fb, axis=0) / np.diff(robot_times[1:])[:, None]
    # window = 15
    # print("Computing moving average")
    # ts_vel_ave = np.zeros_like(ts_vel_fb)
    # ts_acc_ave = np.zeros_like(ts_acc_fb)
    # # fmt: off
    # ts_vel_ave[:, 0] = np.convolve(ts_vel_fb[:, 0], np.ones(window) / window, mode="same")
    # ts_vel_ave[:, 1] = np.convolve(ts_vel_fb[:, 1], np.ones(window) / window, mode="same")
    # ts_vel_ave[:, 2] = np.convolve(ts_vel_fb[:, 2], np.ones(window) / window, mode="same")
    # ts_acc_ave[:, 0] = np.convolve(ts_acc_fb[:, 0], np.ones(window) / window, mode="same")
    # ts_acc_ave[:, 1] = np.convolve(ts_acc_fb[:, 1], np.ones(window) / window, mode="same")
    # ts_acc_ave[:, 2] = np.convolve(ts_acc_fb[:, 2], np.ones(window) / window, mode="same")
    # # fmt: on
    # ts_vel_fb = ts_vel_ave
    # ts_acc_fb = ts_acc_ave

    # Compute velocity and acc with numerial differentiation
    N = ts_pose_fb.shape[0]
    ts_vel_fb = finite_difference(ts_pose_fb[:, :3], 5 + np.arange(N - 10), 0.002, 1)
    ts_acc_fb = finite_difference(ts_pose_fb[:, :3], 5 + np.arange(N - 10), 0.002, 2)

    # compute wrench diff
    dn = 140  # 10ms
    wrench_diff = wrench[dn:] - wrench[:-dn]
    wrench_diff_times = wrench_times[dn:]
    wrench_diff_norm = np.linalg.norm(wrench_diff[:, :3], axis=1)

    ##
    ## compute K2 from velocity resolved law
    ##
    # # big diff id
    # bd_wrench_id = np.squeeze(np.array(np.where(wrench_diff_norm > 1)))
    # # remove entries in id that is smaller than 5
    # bd_wrench_id = bd_wrench_id[5:]
    # # find velocity_id that has time corresponding to bd_wrench_id
    # vel_id = np.searchsorted(robot_times[5:-6], wrench_diff_times[bd_wrench_id])
    # dk = 5
    # V1 = ts_vel_fb[vel_id - 5 - dk, :]
    # V2 = ts_vel_fb[vel_id - 5 + dk, :]
    # V1V2 = V1 - V2
    # F1 = wrench[bd_wrench_id - 5 - dk, :]
    # F2 = wrench[bd_wrench_id - 5 + dk, :]
    # F1F2 = F1 - F2
    # # Project each wrench difference to the velocity difference
    # F1F2 = F1F2[:, :3]
    # V1V2 = V1V2[:, :3]
    # F1F2_norm = np.linalg.norm(F1F2, axis=1)
    # V1V2_proj_norm = np.sum(F1F2 * V1V2, axis=1) / F1F2_norm
    # K2 = V1V2_proj_norm / F1F2_norm
    # K2_times = robot_times[vel_id - 5]
    # fig = make_subplots(
    #     rows=2, cols=1,
    #     shared_xaxes=True, subplot_titles=('K2', 'Fx'))
    # fig.add_trace(go.Scatter(x=K2_times, y=K2, name='K2'),row=1, col=1)
    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench[:,0], name='wrench0'),row=2, col=1)

    ##
    ## Compute a1 a2 b0 b1 b2 from discrete system
    ##
    r_horizon = 50  # 50 number of robot points for one inference, at least 7
    dim = 6  # 3 or 6
    # big diff id
    wrench_aligned_id = np.searchsorted(wrench_times, robot_times)
    wrench_aligned = wrench[wrench_aligned_id - 1, :dim]
    dn = 5  # 10ms
    wrench_aligned_diff_norm = np.linalg.norm(
        wrench_aligned[dn:] - wrench_aligned[:-dn], axis=1
    )

    bd_id = np.squeeze(np.array(np.where(wrench_aligned_diff_norm > 1.5)))
    bd_robot_times = robot_times[bd_id - 1]
    print("len(bd_id): ", len(bd_id))
    # process piece by piece
    a1_all = np.zeros((len(bd_id) - 1))
    a2_all = np.zeros((len(bd_id) - 1))
    b1_all = np.zeros((len(bd_id) - 1))
    b2_all = np.zeros((len(bd_id) - 1))
    for i in tqdm(range(len(bd_id) - 1)):
        ri = bd_id[i]
        if ri >= len(ts_pose_fb) - r_horizon:
            break
        # compute relative pose
        SE3_temp = SE3_fb[ri : ri + r_horizon]
        SE30_inv = SE3_inv(SE3_temp[0])
        SE3_piece = SE30_inv @ SE3_temp

        if dim == 3:
            twist_piece = SE3_piece[:r_horizon, :3, 3]
        else:
            twist_piece = np.zeros((r_horizon, dim))
            for j in range(r_horizon):
                twist_piece[j, :] = SE3_to_spt(SE3_piece[j, :], 1e-15)
        wrench_piece = wrench_aligned[ri : ri + r_horizon, :dim]
        ## Step one: compute the primary direction of variation of this piece
        wrench_piece_delta = np.diff(wrench_piece, axis=0)
        max_id = np.argmax(np.linalg.norm(wrench_piece_delta, axis=1))
        vector = wrench_piece[0] - wrench_piece[max_id + 1]

        if np.linalg.norm(vector) < 1e-4:
            print("Zero vector")
            continue
        vector = vector / np.linalg.norm(vector)
        ## Step two: compute the projection of pos wrench on this vector
        twist_proj = np.sum(twist_piece * vector, axis=1)
        wrench_proj = np.sum(wrench_piece * vector, axis=1)
        # Step three: solve for a1, a2, b0, b1, b2
        # a1, a2, b1, b2 = fit_polynomial_a1a2b1b2(wrench_proj, twist_proj, method="least_square")
        # a1_all[i] = a1
        # a2_all[i] = a2
        # b1_all[i] = b1
        # b2_all[i] = b2
        k = 20
        para = fit_polynomial_k(wrench_proj, twist_proj, k)

        # step four: augment data in the projected space
        # Faug = [0]
        Faug = [0, 1, 2, 10]
        # wrench_proj_aug_all, twist_proj_aug_all = augment_a1a2b1b2(
        #     a1, a2, b1, b2,
        #     wrench_proj,
        #     twist_proj,
        #     Faug,
        # )
        wrench_proj_aug_all, twist_proj_aug_all = augment_k(
            para,
            k,
            wrench_proj,
            twist_proj,
            Faug,
        )
        variation_score = compute_variation_score(wrench_proj)

        # illustrate the result
        # print(f"a1:{a1}, a2:{a2}, b1:{b1}, b2:{b2}.")
        print(f"Score = {variation_score}.")
        c = input("Press Enter to continue, d + Enter to draw")
        if c == "d":
            fig = make_subplots(
                rows=2, cols=1, shared_xaxes=True, subplot_titles=("pos", "force")
            )
            fig.add_trace(
                go.Scatter(
                    x=np.arange(r_horizon),
                    y=twist_proj,
                    name="twist_proj",
                    mode="markers",
                ),
                row=1,
                col=1,
            )
            for f in range(len(Faug)):
                fig.add_trace(
                    go.Scatter(
                        x=np.arange(r_horizon),
                        y=twist_proj_aug_all[f],
                        name=f"twist_proj_aug_{f}",
                        mode="markers",
                    ),
                    row=1,
                    col=1,
                )
            fig.add_trace(
                go.Scatter(
                    x=np.arange(r_horizon),
                    y=wrench_proj,
                    name="wrench_proj",
                    mode="markers",
                ),
                row=2,
                col=1,
            )
            for f in range(len(Faug)):
                fig.add_trace(
                    go.Scatter(
                        x=np.arange(r_horizon),
                        y=wrench_proj_aug_all[f],
                        name=f"wrench_proj_aug_{f}",
                        mode="markers",
                    ),
                    row=2,
                    col=1,
                )
            fig.update_layout(height=600, width=800, title_text=i)
            fig.show()

    fig = make_subplots(
        rows=6,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("a1", "a2", "b1", "b2", "Fx", "Fd"),
    )
    fig.add_trace(
        go.Scatter(
            x=bd_robot_times,
            y=a1_all,
            name="a1",
            mode="markers",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=bd_robot_times,
            y=a2_all,
            name="a2",
            mode="markers",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=bd_robot_times,
            y=b1_all,
            name="b1",
            mode="markers",
        ),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=bd_robot_times,
            y=b2_all,
            name="b2",
            mode="markers",
        ),
        row=4,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=wrench_times, y=wrench[:, 0], name="wrench0"), row=5, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bd_robot_times,
            y=wrench_aligned_diff_norm[bd_id],
            name="big diffs",
            mode="markers",
        ),
        row=6,
        col=1,
    )
    fig.update_layout(height=900, width=1500, title_text=ep)
    fig.show()
    break

    # fig = make_subplots(
    #     rows=4, cols=3,
    #     shared_xaxes=True, subplot_titles=('X', 'Y', 'Z',
    #                                         'dX', 'dY', 'dZ',
    #                                         'ddX', 'ddY', 'ddZ',
    #                                        'fx', 'fy', 'fz'))

    # fig.add_trace(go.Scatter(x=robot_times, y=ts_pose_fb[:,0], name='ts_pose_fb0'),row=1, col=1)
    # fig.add_trace(go.Scatter(x=robot_times, y=ts_pose_fb[:,1], name='ts_pose_fb1'),row=1, col=2)
    # fig.add_trace(go.Scatter(x=robot_times, y=ts_pose_fb[:,2], name='ts_pose_fb2'),row=1, col=3)

    # fig.add_trace(go.Scatter(x=robot_times[1:], y=ts_vel_fb[:,0], name='ts_vel_fb0'),row=2, col=1)
    # fig.add_trace(go.Scatter(x=robot_times[1:], y=ts_vel_fb[:,1], name='ts_vel_fb1'),row=2, col=2)
    # fig.add_trace(go.Scatter(x=robot_times[1:], y=ts_vel_fb[:,2], name='ts_vel_fb2'),row=2, col=3)

    # fig.add_trace(go.Scatter(x=robot_times[2:], y=ts_acc_fb[:,0], name='ts_acc_fb0'),row=3, col=1)
    # fig.add_trace(go.Scatter(x=robot_times[2:], y=ts_acc_fb[:,1], name='ts_acc_fb1'),row=3, col=2)
    # fig.add_trace(go.Scatter(x=robot_times[2:], y=ts_acc_fb[:,2], name='ts_acc_fb2'),row=3, col=3)

    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench[:,0], name='wrench0'),row=4, col=1)
    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench[:,1], name='wrench1'),row=4, col=2)
    # fig.add_trace(go.Scatter(x=wrench_times, y=wrench[:,2], name='wrench2'),row=4, col=3)

    # fig.update_layout(height=900, width=1900, title_text=ep)
    # fig.show()
    # break
    # input("Press Enter to continue...")